# HyperParameters Optimization, 2 steps Neural Network, 2 input parameters

### PROBLEMI: dimensionalità array e performance

In [1]:
import keras.backend as K
from keras.regularizers import l2
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
from keras.models import save_model
import numpy as np
from itertools import product
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from keras.optimizers import Adam, Nadam, Adamax
from ann_functions3D import getModel, kCrossVal, transfBestparam, import_data
from time import perf_counter
import pandas
import pickle
import os

seed = 7
np.random.seed(seed)

c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Data Preparation

In [2]:
########################     PREPARATION      ##########################
# introduction of the data
file_path_LF = os.path.join("..", "..", "Diffusion\DATA", "reaction_diffusion_LF_46_d75.mat")
#file_path_LF = "..\..\Diffusion\DATA\reaction_diffusion_LF_46_d75.mat"
(reaction_LF_test, U_LF_test, x_LF_test) = import_data(file_path_LF)
file_path_HF = os.path.join("..", "..", "Diffusion\DATA", "reaction_diffusion_HF.mat")
(reaction_HF_test, U_HF_test, x_HF_test) = import_data(file_path_HF)

MemoryError: Unable to allocate 2.98 GiB for an array with shape (50, 801, 100, 100) and data type float64

In [ ]:
########################     NORMALIZATION  #########################
# Input

reaction_max = np.max(reaction_LF_test)
reaction_min = np.min(reaction_LF_test)

reaction_LF_test = (reaction_LF_test - reaction_min) / (
    reaction_max - reaction_min
)
reaction_HF_test = (reaction_HF_test - reaction_min) / (
    reaction_max - reaction_min
)


In [4]:
#########################     TRAIN SET      ##########################
NepoLF = 5000  # number of epochs for first NN: NN_LF
NepoHF = 3000  # number of epochs for second NN: NN_HF

#X = np.tile(x_LF_test, int(len(reaction_LF_test)/len(x_LF_test)))
Nlf = 20

permutation1 = np.random.permutation(len(reaction_LF_test))
permutation2 = np.random.permutation(len(x_LF_test))

reaction_LF = reaction_LF_test[permutation1][0:Nlf]
x_LF = x_LF_test[permutation2][0:Nlf]

#reaction_LF = np.column_stack((reaction_LF, X))
reaction_LF_test = np.array(list(product(reaction_LF_test.flatten(), x_LF_test.flatten())))

reaction_LF= np.array(list(product(reaction_LF.flatten(), x_LF.flatten())))
permutation= np.array(list(product(permutation1.flatten(), permutation2.flatten())))


U_LF_test = U_LF_test[
    :,
     - 1,
    :,12
]

In [5]:
# TRANSFORMATION
U_t_max_test = np.max(U_LF_test)
U_t_min_test = np.min(U_LF_test)

U_LF_test = (U_LF_test - U_t_min_test) / (U_t_max_test - U_t_min_test)
#U_HF = (U_HF - U_h_min_test) / (U_h_max_test - U_h_min_test)
#U_LF = U_LF_test[permutation1[0:Nlf],permutation2[0:Nlf]]
U_LF = U_LF_test[permutation]

##
row, col = U_LF_test.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_LF_test = U_LF_test.flatten()[comb]
##
permutation1 = np.random.permutation(len(reaction_HF_test))
permutation2 = np.random.permutation(len(x_HF_test))
n_HF = 15
reaction_HF = reaction_HF_test[permutation1][0:n_HF]

x_HF = x_HF_test[permutation2][0:n_HF]
reaction_HF= np.array(list(product(reaction_HF.flatten(), x_HF.flatten())))
permutation= np.array(list(product(permutation1.flatten(), permutation2.flatten())))
#reaction_HF = np.column_stack((reaction_HF, x_HF))
U_HF_test = U_HF_test[:, -1, :,44]

reaction_HF_test = np.array(list(product(reaction_HF_test.flatten(), x_HF_test.flatten())))

# TRANSFORMATION
U_h_max_test = np.max(U_HF_test)
U_h_min_test = np.min(U_HF_test)
#U_HF = (U_HF - U_h_min_test) / (U_h_max_test - U_h_min_test)
U_HF_test = (U_HF_test - U_h_min_test) / (U_h_max_test - U_h_min_test)

##
#U_HF = U_HF_test[permutation1[0:n_HF],permutation2[0:n_HF]]
U_HF=U_HF_test[permutation]
row, col = U_HF_test.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_HF_test = U_HF_test.flatten()[comb]
##

IndexError: index 80 is out of bounds for axis 0 with size 50

In [ ]:
##########################       FIRST NN: NN_LF     ##########################
K.clear_session()
bestLF_params = {
    "lr": 0.0255,
    "kernel_init": "glorot_uniform",
    "opt": "Adam",
}  # obtained

modelLF = getModel(bestLF_params, "LF")
histLF = modelLF.fit(
    reaction_LF, U_LF, epochs=NepoLF, batch_size=Nlf, verbose=0
)
print("LF NN done")

ULF = modelLF.predict(reaction_LF_test)
print("\nLF Model:")

test_mse = np.mean(np.square(U_LF_test - ULF[:, 0]))
print(f"Test MSE: {test_mse}")

r_2 = 1 - np.sum(np.square(U_LF_test - ULF[:, 0])) / np.sum(
    np.square(U_LF_test - np.mean(U_LF_test))
)
print(f"R^2: {r_2}")

ValueError: Data cardinality is ambiguous:
  x sizes: 400
  y sizes: 20
Make sure all arrays contain the same number of samples.

In [ ]:
start = perf_counter()
##########################    SECOND NN: NN_HF    ##########################
reaction_test_help = modelLF.predict(reaction_HF_test)[:, 0]

reaction_test_in = np.concatenate(
    (reaction_HF_test, reaction_test_help.reshape(-1,1)),axis=1
) # <- TEST INPUT for the second NN: NN_HF

reaction_train_help = modelLF.predict(reaction_HF)[:, 0]  # f_LF(mu_hf_train)
reaction_final = np.concatenate(
    (reaction_HF, reaction_train_help.reshape(-1,1)),axis=1
) # <- TRAINING INPUT for the second NN: NN_HF

name = "2step"

In [ ]:
####################    HYPERPARAMETER OPTIMIZATION    #######################
MAX_EVAL = 20

K.clear_session()
bayes_trials = Trials()
opt_list = ["Adam", "Adamax"]
kernel_list = ["uniform", "glorot_uniform"]
aux_dic = {"opt": opt_list, "kernel_init": kernel_list}
space = {
    "nodes": hp.qloguniform("nodes", np.log(4), np.log(64), 2),
    "l2weight": hp.loguniform("l2weight", np.log(0.0001), np.log(100)),
    "lr": hp.loguniform("lr", np.log(0.0001), np.log(0.1)),
    "kernel_init": hp.choice("kernel_init", kernel_list),
    "opt": hp.choice("opt", opt_list),
}


def objective(params):
    K.clear_session()
    CVres = kCrossVal(2,n_HF, NepoHF, reaction_final, U_HF, params, name)

    # mse, r_squared = calculate_metrics(Nhf, NepoHF, mu_final, U_hf_train, params, name)   # ADDED

    # return {'loss': CVres, 'mse': mse, 'r_squared': r_squared, 'params': params, 'status': STATUS_OK}
    return {"loss": CVres, "params": params, "status": STATUS_OK}


best_params = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=MAX_EVAL,
    trials=bayes_trials,
)


In [ ]:
transfBestparam(best_params, aux_dic)
print(best_params)

####################    NN_HF training and PREDICTION    #######################
finalModel = getModel(
    best_params, name
)  # final model chosen according to the best paramters
hist = finalModel.fit(
    reaction_final,
    U_HF,
    validation_data=(reaction_test_in, U_HF_test),
    epochs=NepoHF,
    batch_size=n_HF,
    verbose=0,
    validation_freq=20,
)

UHF = finalModel.predict(reaction_test_in)

stop = perf_counter()
elapsed = stop - start
print("Elapsed time: ", elapsed)
print("\nHF Model:")

test_mse = np.mean(np.square(U_HF_test - UHF[:, 0]))
print(f"Test MSE: {test_mse}")

r2_HF = 1 - np.sum(np.square(U_HF_test - UHF[:, 0])) / np.sum(
    np.square(U_HF_test - np.mean(U_HF_test))
)
print(f"R^2: {r2_HF}")

#print("Number of basis functions: ", int(Nlf_models[m]))
print("Number of HF data: ", n_HF)

In [ ]:
####################    TRAINING INSIGHTS    #######################
plt.figure()
plt.subplot(2, 1, 1)
plt.plot(hist.history["mse"], color="red", label="High fidelity train mse")
plt.plot(histLF.history["mse"], color="black", label="Low fidelity")
plt.legend()
plt.yscale("log")
plt.subplot(2, 1, 2)
plt.plot(hist.history["val_mse"], color="red")
plt.yscale("log")
plt.show()

In [ ]:
#################################BRUTTA##################################################
####################################################################################


import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Crea un grafico 3D
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Disegna i grafici di dispersione 3D e ottieni i Path3DCollection
scatter1 = ax.scatter(reaction_HF_test[:,0], reaction_HF_test[:,1], UHF, c='r', marker='o', label='estimated $U_{HF}$ by $NN_{HF}$')
scatter2 = ax.scatter(reaction_HF[:,0], reaction_HF[:,1], U_HF, c='b', marker='^', label='HF data')
scatter3 = ax.scatter(reaction_HF_test[:,0], reaction_HF_test[:,1], U_HF_test, c='b', marker='^', label='exact solution')
scatter4 = ax.scatter(reaction_LF_test[:,0], reaction_LF_test[:,1], modelLF.predict(reaction_LF_test), c='b', marker='^', label='estimated $U_{LF}$ by $NN_{LF}$')

# Crea la legenda utilizzando i Path3DCollection
ax.legend(handles=[scatter1, scatter2,scatter3,scatter4])

# Aggiungi etichette agli assi
ax.set_xlabel('mu')
ax.set_ylabel('x')
ax.set_zlabel('U')

# Mostra il grafico
plt.show()
